# ALMA Data Reduction Tutorial

In this tutorial, you will:
- image a real ALMA visibility dataset: a continuum map, then a spectral cube, and do some preliminary analysis.
- make a spectral-line profile, moment-0/-8 maps (Python), and a moment-1 map (CASA)
- (Part 2) compare line fluxes across a few lines observed toward the same target
- The data used, is identical to those published in: https://arxiv.org/abs/2504.17639 and https://arxiv.org/abs/2401.01871. 

## 0. Setup

Install CASA6 and `gdown` to fetch the demo data from Google Drive.

In [ ]:
%%capture
%pip install casatools casatasks casadata casaplotms gdown

`plotms` (used in 1.1 for a uv-amplitude plot) is Qt-based and needs a display, even
when just writing a plot to a file. Colab has none, so start a virtual one:

In [ ]:
%%capture
!apt-get install -y xvfb
%pip install pyvirtualdisplay

from pyvirtualdisplay import Display
_display = Display(visible=0, size=(1024, 768))
_display.start()

CASA needs a small reference-data package before it can be imported. The normal
runtime download can be unreliable in Colab, so we point it at `casadata` (a pip
package bundling the same data) instead:

In [ ]:
import os
import casadata

os.makedirs(os.path.expanduser("~/.casa"), exist_ok=True)
with open(os.path.expanduser("~/.casa/config.py"), "w") as f:
    f.write(f'measurespath = "{casadata.datapath}"\n')
    f.write("measures_auto_update = False\n")
    f.write("data_auto_update = False\n")

print("CASA data ready:", casadata.datapath)


Download the demo visibility data (replace `DRIVE_FOLDER_URL` if it changes):

In [ ]:
import gdown, glob, os, zipfile

DRIVE_FOLDER_URL = "https://drive.google.com/drive/folders/1BKiGlo9c2LY6AA5GPqFOMqArPFdSt1Tt?usp=sharing"

os.makedirs("data", exist_ok=True)
gdown.download_folder(url=DRIVE_FOLDER_URL, output="data", quiet=False, use_cookies=False)

zips = glob.glob("data/**/*.zip", recursive=True)
print("found zip(s):", zips)
with zipfile.ZipFile(zips[0]) as z:
    z.extractall("data")

ms_dirs = glob.glob("data/**/*.ms", recursive=True)
vis = ms_dirs[0]
print("using MS:", vis)


## Part 1 — imaging a single visibility dataset

### 1.1 Orientation: what's in this Measurement Set?

In [ ]:
from casatasks import listobs

listobs(vis=vis, listfile="listobs.txt", overwrite=True)
!cat listobs.txt


Amplitude vs. uv-distance — a quick look at how the signal falls off with baseline length:

In [ ]:
import os
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"   # casaplotms ships
    # protobuf code generated against an older protoc; newer protobuf installs
    # otherwise raise "Descriptors cannot be created directly" on import.

from casaplotms import plotms
from IPython.display import Image

plotms(vis=vis, xaxis="uvdist", yaxis="amp",
       plotfile="uvdist_amp.png", overwrite=True, showgui=False,
       dpi=150, width=900, height=600)
Image("uvdist_amp.png")


A few things to note in the listing above:
- **Fields**: the sky positions observed.
- **Spectral windows (spw)**: one spw here, already trimmed down to just the
  channels covering our line.
- **Scans**: separate observing blocks; tasks like `tclean`/`mstransform` can select
  scans, spws, fields, etc. independently.

The line here is CO(12-11), observed around **246.75 GHz** at this target's redshift.
Keep that number in mind for the spectrum in step 1.4.

### 1.2 `tclean`: continuum image

A few of the parameters below are worth understanding rather than just copying:

- **`cell`** (pixel size) should be small enough to properly sample the resolution set
  by your longest baseline. Resolution (in arcsec) is roughly
  `206265 * (wavelength / max_baseline_m)`, and you want about 4 to 5 pixels across
  that resolution element. You can read the max baseline straight off the uv-distance
  plot in 1.1 (the x-axis, in meters) — here it maxes out around 700 m, which at about
  247 GHz gives a resolution of about 0.35 arcsec, so `cell="0.05arcsec"` (about 7
  pixels across it) is a reasonable choice.
- **`imagename`** is just the output filename prefix — `tclean` writes `cont.image`,
  `cont.psf`, `cont.pb`, etc.
- **`specmode="mfs"`** (multi-frequency synthesis) combines every channel in the spw
  into a single continuum image, instead of imaging each channel separately.
- **`niter=0`** means zero cleaning iterations: this is a "dirty" image (just the
  inverse Fourier transform of the visibilities, no deconvolution). It's fast and fine
  for a first look, but a real continuum measurement usually needs a *cleaned* image —
  set `niter` to a few thousand, add a `threshold`, and consider `deconvolver`/`mask`.
  See the [tclean docs](https://casadocs.readthedocs.io/en/stable/api/tt/casatasks.imaging.tclean.html)
  for the full parameter list.

In [ ]:
from casatasks import tclean, exportfits

os.system("rm -rf cont.*")
tclean(vis=vis, imagename="cont", specmode="mfs",
       imsize=512, cell="0.05arcsec", weighting="natural",
       niter=0, pblimit=0.0)
exportfits("cont.image", "cont.fits", overwrite=True)


### 1.3 `tclean`: spectral cube

Same field, but `specmode='cube'` keeps every channel separate instead of averaging
them.

Note we pass the *observed* line frequency to `restfreq`, not its rest-frame value —
that keeps the velocity axis (used later for the moment-1 map) centered near zero
instead of showing the line's raw cosmological redshift.

In [ ]:
os.system("rm -rf cube.*")
tclean(vis=vis, imagename="cube", specmode="cube",
       imsize=512, cell="0.05arcsec", weighting="natural",
       niter=0, pblimit=0.0,
       restfreq="246.75GHz", outframe="LSRK")
exportfits("cube.image", "cube.fits", overwrite=True)


### 1.4 Plot the continuum map

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits

cont_hdu = fits.open("cont.fits")[0]
cont_map = np.squeeze(cont_hdu.data)

plt.figure(figsize=(6, 5))
im = plt.imshow(cont_map * 1e3, origin="lower", cmap="inferno")
plt.colorbar(im, label="mJy/beam")
plt.title("Continuum map")
plt.xlabel("pixel"); plt.ylabel("pixel")
plt.show()


### 1.5 Plot the spectral line profile

Spectrum at the map center (the source sits at the phase center here).

In [ ]:
cube_hdul = fits.open("cube.fits")
cube = np.squeeze(cube_hdul[0].data)          # (nchan, ny, nx)
header = cube_hdul[0].header

nchan, ny, nx = cube.shape
crval3, cdelt3, crpix3 = header["CRVAL3"], header["CDELT3"], header["CRPIX3"]
freq_hz = crval3 + (np.arange(nchan) - (crpix3 - 1)) * cdelt3

cy, cx = ny // 2, nx // 2
spectrum_mjy = cube[:, cy, cx] * 1e3

plt.figure(figsize=(10, 4))
plt.step(freq_hz / 1e9, spectrum_mjy, where="mid", color="steelblue")
plt.axhline(0, color="gray", lw=0.6, ls="--")
plt.axvline(246.75, color="crimson", lw=0.8, ls=":", label="predicted line center")
plt.xlabel("Frequency [GHz]"); plt.ylabel("Flux density [mJy/beam]")
plt.title("Spectrum at map center")
plt.legend()
plt.show()


### 1.6 Moment-0 and moment-8 maps (pure Python)

- **Moment-8**: peak signal-to-noise across the spectrum at each pixel — a quick
  "where's the line" map.
- **Moment-0**: integrated line flux, summed over the channels containing the line
  (read off the spectrum above; adjust `line_lo`/`line_hi` if yours differs).

In [ ]:
line_lo, line_hi = 110, 200   # <-- adjust after inspecting your own spectrum plot

stds = np.nanstd(cube, axis=(1, 2))                  # per-channel RMS across the map
rms = stds.copy()
rms[(rms == 0) | np.isnan(rms)] = np.nan
snr_cube = cube / rms[:, np.newaxis, np.newaxis]
moment8 = np.nanmax(snr_cube, axis=0)                # peak-SNR map

chan_width_hz = abs(cdelt3)
restfreq_hz = 246.75e9   # same observed-frame value used for tclean's restfreq
dv_kms = chan_width_hz / restfreq_hz * 2.99792458e5
moment0 = np.nansum(cube[line_lo:line_hi], axis=0) * dv_kms * 1e3   # mJy/beam * km/s

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
im0 = axes[0].imshow(moment0, origin="lower", cmap="inferno")
axes[0].set_title("Moment-0 (integrated flux)")
plt.colorbar(im0, ax=axes[0], label="mJy/beam km/s", shrink=0.85)

im1 = axes[1].imshow(moment8, origin="lower", cmap="inferno")
axes[1].set_title("Moment-8 (peak SNR)")
plt.colorbar(im1, ax=axes[1], label="SNR", shrink=0.85)
plt.tight_layout()
plt.show()


### 1.7 Moment-1 map (the CASA way)

Moment-1 (intensity-weighted velocity) needs a flux threshold to avoid being dominated
by noise — CASA's `immoments` handles that natively.

In [ ]:
from casatasks import immoments

os.system("rm -rf cube.mom1*")
immoments(imagename="cube.image", moments=[1],
          chans=f"{line_lo}~{line_hi - 1}",
          includepix=[3 * float(np.nanmedian(stds)), 1e9],   # crude 3-sigma-ish mask
          outfile="cube.mom1")
exportfits("cube.mom1", "cube.mom1.fits", overwrite=True)

mom1 = np.squeeze(fits.open("cube.mom1.fits")[0].data)
plt.figure(figsize=(6, 5))
im = plt.imshow(mom1, origin="lower", cmap="RdBu_r")
plt.colorbar(im, label="km/s")
plt.title("Moment-1 (velocity field)")
plt.show()


## Part 2 — comparing multiple lines for the same target

This target has been observed in several other ALMA bands too, each covering a
different line. This section reads in cubes for those lines as they become available
and compares integrated fluxes.

We already have a second one: a Band 10 cube with a line that identifies as [OI]63um
at this target's redshift. Its FITS file is in the same shared Drive folder as the
Band 6 data, so it was already fetched back in Setup.

> Add more lines to `LINE_CUBES` below as their cubes are exported and uploaded to the
> same Drive folder.

In [ ]:
oi63_matches = glob.glob("data/**/WISE2246-0526_12m_838GHz_tbin30s_cube.cube.fits", recursive=True)
print("found:", oi63_matches)
oi63_path = oi63_matches[0]

LINE_CUBES = {
    "CO(12-11)": "cube.fits",     # from Part 1, already in hand
    "[OI]63um":  oi63_path,       # Band 10, downloaded in Setup
    # "[CII]158um":  "data/.../cube.fits",
    # "[NII]205um":  "data/.../cube.fits",
    # ... add more as they become available
}

REST_GHZ = {
    "CO(12-11)": 1381.995,
    "[OI]63um": 4744.777490, "[OI]145um": 2060.069000,
    "[NII]122um": 2459.380000, "[NII]205um": 1461.133800,
    "[OIII]88um": 3393.006240, "[CII]158um": 1900.536900,
    "[CI]370um": 809.341970, "[CI]609um": 492.160651,
}
Z_SYSTEMIC = 4.601


In [ ]:
def integrated_flux_mjy_kms(fits_path, rest_ghz, z=Z_SYSTEMIC, pos=None, half_width_chan=20):
    """Sum flux around the map center over +/- half_width_chan channels centered
    on the peak-SNR channel -- a simplified version of Part 1's moment-0, applied
    consistently to every line's cube. Uses the observed-frame frequency for the
    Hz->km/s conversion, matching how these cubes were imaged.
    """
    hdul = fits.open(fits_path)
    c = np.squeeze(hdul[0].data)
    hdr = hdul[0].header
    nchan = c.shape[0]
    cdelt3 = hdr["CDELT3"]

    stds = np.nanstd(c, axis=(1, 2))
    rms = stds.copy(); rms[(rms == 0) | np.isnan(rms)] = np.nan

    y, x = pos if pos else (c.shape[1] // 2, c.shape[2] // 2)
    spec = c[:, y, x] / rms
    peak_ch = int(np.nanargmax(spec))
    lo, hi = max(0, peak_ch - half_width_chan), min(nchan, peak_ch + half_width_chan)

    obs_ghz = rest_ghz / (1 + z)
    dv_kms = abs(cdelt3) / (obs_ghz * 1e9) * 2.99792458e5
    return float(np.nansum(c[lo:hi, y, x]) * dv_kms * 1e3)   # mJy/beam * km/s


fluxes = {name: integrated_flux_mjy_kms(path, REST_GHZ[name])
          for name, path in LINE_CUBES.items()}
print(fluxes)


In [ ]:
names = list(fluxes.keys())
vals = [fluxes[n] for n in names]
ref = names[0]   # normalize ratios to the first available line

plt.figure(figsize=(7, 4))
plt.bar(names, [v / fluxes[ref] for v in vals], color="steelblue")
plt.ylabel(f"Line flux / {ref}")
plt.title("Line ratios")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()
